In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import random
import numpy as np

# Device configuration (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# CNN model with variable architecture defined by Firefly Algorithm
class CNN(nn.Module):
    def __init__(self, conv_layers, fc_units):
        super(CNN, self).__init__()
        layers = []
        in_channels = 3

        # Build convolutional layers dynamically
        for out_channels, kernel_size in conv_layers:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_channels = out_channels

        self.conv = nn.Sequential(*layers)

        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(in_channels * 4 * 4, fc_units),
            nn.ReLU(),
            nn.Linear(fc_units, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Evaluate model accuracy on test set
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

# Firefly Algorithm for CNN architecture optimization
class FireflyAlgorithm:
    def __init__(self, n_fireflies, max_iter):
        self.n_fireflies = n_fireflies
        self.max_iter = max_iter
        self.population = [self.initialize_firefly() for _ in range(n_fireflies)]

    # Initialize random architecture
    def initialize_firefly(self):
        conv_layers = [(random.choice([16, 32, 64]), 3) for _ in range(random.randint(2, 4))]
        fc_units = random.choice([128, 256, 512])
        return {'conv_layers': conv_layers, 'fc_units': fc_units, 'accuracy': 0}

    # Move fireflies toward better solutions
    def move_fireflies(self):
        for i in range(self.n_fireflies):
            for j in range(self.n_fireflies):
                if self.population[j]['accuracy'] > self.population[i]['accuracy']:
                    self.population[i] = self.modify_firefly(self.population[i], self.population[j])

    # Modify architecture based on better firefly
    def modify_firefly(self, firefly, best_firefly):
        for idx in range(len(firefly['conv_layers'])):
            if random.random() < 0.5:
                firefly['conv_layers'][idx] = best_firefly['conv_layers'][idx]

        firefly['fc_units'] = best_firefly['fc_units'] if random.random() < 0.5 else firefly['fc_units']
        return firefly

    # Optimization process
    def optimize(self):
        for _ in range(self.max_iter):
            for firefly in self.population:

                # Build model based on current architecture
                model = CNN(firefly['conv_layers'], firefly['fc_units']).to(device)

                optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
                criterion = nn.CrossEntropyLoss()

                # Short training phase
                model.train()
                for images, labels in trainloader:
                    images, labels = images.to(device), labels.to(device)

                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

                # Evaluate accuracy
                firefly['accuracy'] = evaluate_model(model)

            # Move fireflies
            self.move_fireflies()

        # Return best architecture
        best_firefly = max(self.population, key=lambda x: x['accuracy'])
        return best_firefly

# Run Firefly Optimization
fa = FireflyAlgorithm(n_fireflies=5, max_iter=10)
best_architecture = fa.optimize()

print("Best architecture found:", best_architecture)